# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from itertools import product
import geopandas as gpd
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm

root_path = Path(globals()['_dh'][0]).resolve().parent
sys.path.append(str(root_path))

from paths import config_path, input_path, api_path
from library.utilities import get_path

In [3]:
# 2. Load the configuration

with (config_path / 'municipality-prototype.json').open('r', encoding='utf-8') as file:
    config = json.load(file)

In [10]:
# 3. Load the base demand

base_demand = pd.read_csv(input_path / config['loader']['properties']['access'] / 'base_demand' / get_path(config['loader']['name'],config['loader']['properties'], "csv"), index_col=['timestamp'], parse_dates=['timestamp'])

In [11]:
# 4 Execute transformations - Sort the transformations array

transformers = config['transformers']
transformers.sort(key=lambda x: x["order"])

In [12]:
# 4.1. Execute transformations - Split base demand (national) into geographies (municipalities)

input = {input['name']: pd.read_csv(input_path / transformers[0]['access'] / input['path'], usecols=input['columns'], dtype={input['index']: str}, index_col=[input['index']]) for input in transformers[0]['inputs']}

# Calculate municipal demand
geography_demand_matrix = input['municipality_energy_split']['ratio'].values[:, None] * base_demand['Sweden'].values

# Generate a 10% randomness matrix
randomness = 0.1 # +/- 10%
num_geos, num_hours = geography_demand_matrix.shape
random_factors = np.random.uniform(-randomness, randomness, size=(num_geos, num_hours))

# Apply randomness and normalize
geography_demand_matrix_randomized = geography_demand_matrix * (1 + random_factors)
scaling_factor = geography_demand_matrix.sum(axis=1, keepdims=True) / geography_demand_matrix_randomized.sum(axis=1, keepdims=True)
geography_demand_matrix_randomized *= scaling_factor

# Create new dataframe index (includes geography)
geography_index = pd.MultiIndex.from_product(
    [input['municipality_energy_split'].index, base_demand.index],
    names=['geography', 'timestamp']
)

# Create a new dataframe on the municipal demand with randomness applied
geography_demand = pd.DataFrame(
    geography_demand_matrix_randomized.ravel(),
    index=geography_index, 
    columns=['demand']
)

In [13]:
# 4.2. Execute transformations - Apply growth over time

# Define growth parameters
target_energy = 330  # TWh projected for 2045
yearly_growth = (target_energy / (base_demand['Sweden'].sum() / 1_000_000)) ** (1 / 20) - 1  # About 4.32% growth yearly
twenty_yrs_in_hours = (datetime(2045, 1, 1, 0, 0, 0) - datetime(2025, 1, 1, 0, 0, 0)).total_seconds() / 3600
hourly_growth = (target_energy / (base_demand['Sweden'].sum() / 1_000_000)) ** (1 / twenty_yrs_in_hours) - 1

# Generate the new timestamps for the extended demand
start_time = pd.Timestamp("2025-01-01 00:00:00")
end_time = pd.Timestamp("2045-12-31 23:00:00")
new_timestamps = pd.date_range(start=start_time, end=end_time, freq='h')

# Calculate the growth factors for all hours
hours_elapsed = np.arange(len(new_timestamps))  # Elapsed hours as a 1D array
growth_factors = (1 + hourly_growth) ** hours_elapsed  # Shape: (len(new_timestamps),)

# Prepare the new DataFrame structure
geographies = geography_demand.index.get_level_values('geography').unique()
new_index = pd.MultiIndex.from_product(
    [geographies, new_timestamps], names=["geography", "timestamp"]
)

# Extract hourly demand patterns for 2024
hourly_demand_2024 = (
    geography_demand.loc[
        geography_demand.index.get_level_values('timestamp').year == 2024
    ]
    .reset_index()
)

# Repeat the 2024 demand pattern to match the new timestamps
hourly_demand_pattern = hourly_demand_2024.groupby("geography")["demand"].apply(
    lambda x: np.tile(x.values, len(new_timestamps) // len(x) + 1)[:len(new_timestamps)]
)

# Convert hourly demand pattern back to a 2D array (municipalities x timestamps)
base_demand_repeated = np.vstack(hourly_demand_pattern.values)

# Generate randomness
num_geos, num_hours = base_demand_repeated.shape
random_factors = np.random.uniform(-randomness, randomness, size=(num_geos, num_hours))


# Apply growth factors
extended_demand_values = base_demand_repeated * growth_factors * (1+random_factors)  # Shape: (num_municipalities, len(new_timestamps))

# Create the extended demand DataFrame
extended_geography_demand = pd.DataFrame(
    data=extended_demand_values.flatten(),
    index=new_index,
    columns=["demand"]
)

In [14]:
# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']

## TODO: Make this less naive

## This transform is not realistic. It does the following:
##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality
##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder 

# Extract July data for each year
july_data = extended_geography_demand.loc[
    extended_geography_demand.index.get_level_values('timestamp').month == 7
]

# Compute the lowest July value per municipality and year
lowest_july_per_year = (
    july_data
    .groupby([july_data.index.get_level_values('geography'),
              july_data.index.get_level_values('timestamp').year])['demand']
    .min()
)

# Convert to DataFrame and calculate industry demand
lowest_july_per_year = lowest_july_per_year.to_frame(name='lowest_july')
lowest_july_per_year['industry_demand'] = lowest_july_per_year['lowest_july'] * 0.5

# Create extended_municipal_sector_demand and merge industry demand back with the full extended demand
# Add a column to join by year
extended_geography_sector_demand = extended_geography_demand.copy()
extended_geography_sector_demand['year'] = extended_geography_sector_demand.index.get_level_values('timestamp').year

# Join the calculated industry demand
extended_geography_sector_demand = extended_geography_sector_demand.join(
    lowest_july_per_year['industry_demand'], 
    on=['geography', 'year']
)

# Industry demand is constant across each municipality and year
extended_geography_sector_demand['industry'] = extended_geography_sector_demand['industry_demand']

# Compute the remainder for buildings and transport
extended_geography_sector_demand['remainder'] = extended_geography_sector_demand['demand'] - extended_geography_sector_demand['industry']

# Split remainder into buildings (95%) and transport (5%)
extended_geography_sector_demand['buildings'] = extended_geography_sector_demand['remainder'] * 0.95
extended_geography_sector_demand['transport'] = extended_geography_sector_demand['remainder'] * 0.05

# Drop unnecessary intermediate columns
extended_geography_sector_demand = extended_geography_sector_demand.drop(columns=['industry_demand', 'remainder', 'year'])

# Rename 'demand' to 'total' for clarity
extended_geography_sector_demand = extended_geography_sector_demand.rename(columns={'demand': 'total'})


In [15]:
# 5. Write output

# Write yearly per municipality (1h, 3h, 1d, 1w, 1m, 1y), for the country as a whole

# TODO: This script currently takes an hour+ to run through 20 years and 290 municipalities and 5 resolutions. I need to improve this.

print('Preparing data...')

geos = extended_geography_sector_demand.index.get_level_values('geography').unique()
years = extended_geography_sector_demand.index.get_level_values('timestamp').year.unique()
aggregations = config['output']['properties']['aggregations']

melted_aggregations = [
    {"resolution": item["resolution"], "aggregation": stat}
    for item in aggregations
    for stat in item["aggregation"]
]

Preparing data...


In [9]:
# Write demand summed over geos

print("Aggregating data over municipalities...")

country_data = extended_geography_sector_demand.groupby(level='timestamp').sum()
for year in tqdm(years, desc="Progress (years)", unit="years"):
    yearly_data = country_data.loc[country_data.index.year == year]
    for agg in melted_aggregations:
        if agg['aggregation'] != 'none':
            yearly_data.resample(agg['resolution']).agg(agg['aggregation']).to_csv(
                api_path / f"demand_t,geography=00,resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year}.csv.gz",
                compression='gzip'
            )

Aggregating data over municipalities...


Progress (years): 100%|██████████| 21/21 [00:04<00:00,  4.64years/s]


In [16]:
# Write yearly demand for all geos, per year and aggregation
# Write 

# Find aggregations for 1YE
aggregations_1YE = next(item['aggregation'] for item in aggregations if item['resolution'] == '1YE')

# Group data by year and municipality for all aggregations
print('Aggregating data per year...')
grouped = extended_geography_sector_demand.groupby([extended_geography_sector_demand.index.get_level_values('timestamp').year, 'geography'])
yearly_stats = {stat: grouped.agg(stat) for stat in aggregations_1YE}

# Write all years per municipality
for geo in tqdm(geos, desc="Progress (geos)", unit="geo"):
    for agg in aggregations_1YE:
        geo_data = yearly_stats[agg].xs(geo, level='geography').rename_axis(index={"timestamp": "year"})
        geo_data.to_csv(api_path / f"demand,geography={geo},resolution=1YE,sector=all,aggregation={agg},year=all.csv.gz", compression='gzip')

Aggregating data per year...


Progress (geos): 100%|██████████| 290/290 [00:01<00:00, 154.99geo/s]


In [ ]:
# Write yearly demand for all geos, per year and aggregation
# Write 

# Find aggregations for 1YE
aggregations_1YE = next(item['aggregation'] for item in aggregations if item['resolution'] == '1YE')

# Group data by year and municipality for all aggregations
print('Aggregating data per year...')
grouped = extended_geography_sector_demand.groupby([extended_geography_sector_demand.index.get_level_values('timestamp').year, 'geography'])
yearly_stats = {stat: grouped.agg(stat) for stat in aggregations_1YE}

# Write all municipalities per year
for year in tqdm(years, desc="Progress (years)", unit="years"):
    for agg in aggregations_1YE:
        year_data = yearly_stats[agg].loc[year]
        year_data.loc['00'] = year_data.agg(agg)
        year_data.to_csv(api_path / f"demand,geography=all,resolution=1YE,sector=all,aggregation={agg},year={year}.csv.gz", compression='gzip')

# Write all years per municipality
for geo in tqdm(geos, desc="Progress (geos)", unit="geo"):
    for agg in aggregations_1YE:
        geo_data = yearly_stats[agg].xs(geo, level='geography')
        geo_data.to_csv(api_path / f"demand,geography={geo},resolution=1YE,sector=all,aggregation={agg},year=all.csv.gz", compression='gzip')

# Write all years for the whole country
for agg in aggregations_1YE:
    national_data = yearly_stats['sum'].groupby(level='timestamp').agg(agg)
    national_data.to_csv(api_path / f"demand,geography={'00'},resolution=1YE,sector=all,aggregation={agg},year=all.csv.gz", compression='gzip')

# Write a single file for all years
for agg in tqdm(aggregations_1YE, desc="Progress (aggregations)", unit="aggregation"):
    year_data = yearly_stats['sum'].rename_axis(index={"timestamp": "year"})
    pd.concat([year_data, year_data.groupby(level='year').agg('sum').assign(geography='00').set_index('geography', append=True)]).reorder_levels(['year', 'geography']).sort_index().to_csv(
        api_path / f"demand,geography=all,resolution=1YE,sector=all,aggregation={agg},year=all.csv.gz", compression='gzip')

# Write globals

Aggregating data per year...


Progress (aggregations): 100%|██████████| 3/3 [00:00<00:00,  3.63aggregation/s]


In [16]:
# Write all years for the whole country
for agg in aggregations_1YE:
    national_data = yearly_stats['sum'].groupby(level='timestamp').agg(agg).rename_axis(index={"timestamp": "year"})
    national_data.to_csv(api_path / f"demand,geography={'00'},resolution=1YE,sector=all,aggregation={agg},year=all.csv.gz", compression='gzip')


In [12]:
# Write demand per geo and year

def process_geo_year(task):
    geo, year = task  # Unpack the task tuple
    municipal_data = extended_geography_sector_demand.xs(geo, level='geography')
    yearly_data = municipal_data.loc[municipal_data.index.year == year]
    
    for agg in melted_aggregations:
        if agg['aggregation'] != 'none':
            yearly_data.resample(agg['resolution']).agg(agg['aggregation']).to_csv(
                api_path / f"demand_t,geography={geo},resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year}.csv.gz",
                compression='gzip'
            )
        else:
            yearly_data.to_csv(
                api_path / f"demand_t,geography={geo},resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year}.csv.gz",
                compression='gzip'
            )

# Create tasks as (geo, year) pairs
tasks = list(product(geos, years))

# Use ProcessPoolExecutor with a top-level function
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(process_geo_year, tasks), total=len(tasks), desc="Geo-Year Pairs"))

Geo-Year Pairs: 100%|██████████| 6090/6090 [15:01<00:00,  6.76it/s] 


In [4]:
# Write the geojson file (no energy data in it right now. Just a renaming.)

geographies_gdp = gpd.read_file(input_path / 'public' / 'geographies' / 'georef-sweden-kommun@public.geojson', encoding='utf-8')

# Save the GeoJSON file
output_file = api_path / f"geo,division=municipality.geojson"
geographies_gdp.to_file(output_file, driver="GeoJSON", encoding="utf-8")


In [5]:
geographies_gdp

,geo_point_2d,year,lan_code,lan_name,kom_code,kom_name,kom_area_code,kom_type,geometry
0,"{ ""lon"": 17.239017732460756, ""lat"": 66.3645005...",2022,25,Norrbottens län,2506,Arjeplog,SWE,Kommun,"POLYGON ((16.40245 67.19024, 16.39827 67.15788..."
1,"{ ""lon"": 17.64870387754971, ""lat"": 59.35234036...",2022,01,Stockholms län,0125,Ekerö,SWE,Kommun,"POLYGON ((17.50466 59.44154, 17.48917 59.44126..."
2,"{ ""lon"": 13.478294294409467, ""lat"": 59.8210855...",2022,17,Värmlands län,1762,Munkfors,SWE,Kommun,"POLYGON ((13.6289 59.81485, 13.61831 59.81867,..."
3,"{ ""lon"": 13.367667436573603, ""lat"": 55.8360844...",2022,12,Skåne län,1285,Eslöv,SWE,Kommun,"POLYGON ((13.33851 56.00629, 13.33502 56.00238..."
4,"{ ""lon"": 18.401337385868135, ""lat"": 63.4598749...",2022,22,Västernorrlands län,2284,Örnsköldsvik,SWE,Kommun,"POLYGON ((18.05554 63.18622, 18.18274 63.15991..."
...,...,...,...,...,...,...,...,...,...
285,"{ ""lon"": 18.007859398914995, ""lat"": 59.3686544...",2022,01,Stockholms län,0184,Solna,SWE,Kommun,"POLYGON ((17.98004 59.3928, 17.96765 59.39314,..."
286,"{ ""lon"": 15.785091377399048, ""lat"": 59.3547104...",2022,19,Västmanlands län,1984,Arboga,SWE,Kommun,"POLYGON ((15.7126 59.52588, 15.71851 59.5163, ..."
287,"{ ""lon"": 12.939882869169779, ""lat"": 55.5670869...",2022,12,Skåne län,1280,Malmö,SWE,Kommun,"POLYGON ((12.97081 55.67946, 12.87786 55.66387..."
288,"{ ""lon"": 17.395201933829018, ""lat"": 59.1940356...",2022,01,Stockholms län,0140,Nykvarn,SWE,Kommun,"POLYGON ((17.34157 59.27353, 17.32762 59.27073..."


In [14]:
# Write the parameters.json

print("Writing parameters.json")

geographies = geographies_gdp[['kom_type', 'kom_code', 'kom_name', 'lan_code', 'lan_name']].copy()

geographies = geographies.rename(columns={
    "kom_type": "type",
    "kom_code": "id",
    "kom_name": "name",
    "lan_code": "parent_id",
    "lan_name": "parent_name"
})

## Add Sweden as a whole
new_row = pd.DataFrame({
    "type": ["Land"],
    "id": ["00"],
    "name": ["Sverige"],
    "parent_name": [""],
    "parent_id": [""]
})

geographies = pd.concat([geographies, new_row], ignore_index=True)

parameters = {
    'years': list(range(config['start-year'], config['end-year'])),
    'geographies': geographies.to_dict(orient="records"),
    'aggregations': config['output']['properties']['aggregations'],
    'sectors': config['output']['properties']['sectors']
}

(api_path / "parameters.json").write_text(json.dumps(parameters, indent=4, ensure_ascii=False), encoding='utf-8')


Writing parameters.json


54558